In [1]:
from pathlib import Path
import numpy as np
import torch
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import ConcatDataset
from sklearn.model_selection import StratifiedKFold

In [2]:
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name != "seai_project":
    PROJECT_DIR = PROJECT_DIR / "seai_project"

BASE_DIR = PROJECT_DIR / "dataset"
DATA_ROOT = BASE_DIR / "oxford_pets"
SPLITS_DIR = BASE_DIR / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SEED = 43
N_FOLDS = 5

TEST_MODE = True

if TEST_MODE:
    N_MAJORITY_TRAIN = 100
    N_MINORITY_TRAIN = 20
    N_TEST_PER_CLASS = 40
    RUN_TAG = "TEST"
else:
    N_MAJORITY_TRAIN = 4000
    N_MINORITY_TRAIN = 400
    N_TEST_PER_CLASS = 600
    RUN_TAG = "FULL"

SPLIT_PATH = SPLITS_DIR / f"split_indices_{RUN_TAG}_seed{SPLIT_SEED}.npz"
CV_FOLDS_PATH = SPLITS_DIR / f"cv_folds_{RUN_TAG}_seed{SPLIT_SEED}.npz"

print("Downloading/Verifying dataset...")
base_trainval = OxfordIIITPet(root=DATA_ROOT, split="trainval", download=True)
base_test = OxfordIIITPet(root=DATA_ROOT, split="test", download=True)
full_dataset = ConcatDataset([base_trainval, base_test])

all_targets = np.concatenate([base_trainval._labels, base_test._labels])
classes = base_trainval.classes

cat_class_indices = {classes.index(b) for b in [
    "Abyssinian", "Bengal", "Birman", "Bombay", "British Shorthair",
    "Egyptian Mau", "Maine Coon", "Persian", "Ragdoll", "Russian Blue",
    "Siamese", "Sphynx"
]}

binary_targets = np.array([1 if t in cat_class_indices else 0 for t in all_targets])
majority_indices = np.where(binary_targets == 0)[0]
minority_indices = np.where(binary_targets == 1)[0]

print(f"Available dogs (0): {len(majority_indices)}")
print(f"Available cats (1): {len(minority_indices)}")

rng = np.random.default_rng(SPLIT_SEED)
maj = rng.permutation(majority_indices)
min_ = rng.permutation(minority_indices)

majority_train_idx = maj[:N_MAJORITY_TRAIN]
minority_train_idx = min_[:N_MINORITY_TRAIN]

d0, c0 = N_MAJORITY_TRAIN, N_MINORITY_TRAIN

majority_test_idx = maj[d0:d0 + N_TEST_PER_CLASS]
minority_test_idx = min_[c0:c0 + N_TEST_PER_CLASS]

np.savez(
    SPLIT_PATH,
    majority_train_idx=majority_train_idx,
    minority_train_idx=minority_train_idx,
    majority_test_idx=majority_test_idx,
    minority_test_idx=minority_test_idx
)
print(f"Base split saved to: {SPLIT_PATH}")

train_idx = np.concatenate([majority_train_idx, minority_train_idx])
cv_indices = train_idx.copy()
cv_labels = binary_targets[cv_indices]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
fold_data = {}

for fold, (train_pos, val_pos) in enumerate(skf.split(cv_indices, cv_labels), 1):
    fold_data[f"train_{fold}"] = cv_indices[train_pos]
    fold_data[f"val_{fold}"] = cv_indices[val_pos]

np.savez(CV_FOLDS_PATH, **fold_data)
print(f"5-Fold CV indices saved to: {CV_FOLDS_PATH}")

Downloading/Verifying dataset...


100%|██████████| 792M/792M [00:08<00:00, 94.7MB/s] 
100%|██████████| 19.2M/19.2M [00:00<00:00, 42.7MB/s]


Available dogs (0): 4978
Available cats (1): 2371
Base split saved to: /workspace/seai_project/dataset/splits/split_indices_TEST_seed43.npz
5-Fold CV indices saved to: /workspace/seai_project/dataset/splits/cv_folds_TEST_seed43.npz
